# Week 6 Integrative Capstone — Exploratory Analysis Notebook

This notebook walks through the combined end-to-end Online Retail pipeline:
- Data loading and cleaning
- Exploratory data analysis
- RFM + supervised behavioural feature engineering
- KMeans unsupervised customer segmentation
- Temporal-split supervised future-purchase classification (LR vs RF)
- Honest tabular MLP deep-learning baseline
- Final recommendations

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT.resolve())

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)
from src.config import RAW_DATA_PATH, RANDOM_SEED
print('Dataset path:', RAW_DATA_PATH.resolve(), '| exists =', RAW_DATA_PATH.exists())
print('Random seed:', RANDOM_SEED)

In [ ]:
from src.data_loading import load_excel_dataset, get_dataset_shape
from src.preprocessing import clean_dataset
df_raw = load_excel_dataset(RAW_DATA_PATH)
print('Raw rows x cols:', get_dataset_shape(df_raw))
df_clean, df_positive = clean_dataset(df_raw)
print('Positive transactions w/ CustomerID:', len(df_positive),
      f'({len(df_positive)/len(df_raw)*100:.1f}%)')

In [ ]:
from src.eda import run_eda
eda_summaries = run_eda(df_clean, df_positive)
for k, v in eda_summaries.items():
    print(f'{k}: {v.shape}')

In [ ]:
from src.feature_engineering import build_rfm_features
rfm_df = build_rfm_features(df_positive)
print('RFM rows:', len(rfm_df))
rfm_df.describe().round(2)

In [ ]:
from src.unsupervised import run_unsupervised_pipeline
unsup = run_unsupervised_pipeline(rfm_df)
print('Best k =', unsup['best_k'],
      '| Silhouette =', unsup['clustering_metrics_dict']['silhouette_score'])

In [ ]:
from src.supervised import run_supervised_pipeline
sup = run_supervised_pipeline(df_positive)
m = sup['test_metrics_dict']
print('Best model:', sup['best_model_name'])
print(f'Acc={m["accuracy"]:.3f} | F1={m["f1"]:.3f} | ROC-AUC={m["roc_auc"]:.3f}')

In [ ]:
from src.deep_learning import build_dl_section_summary
dl = build_dl_section_summary(
    X_train=sup.get('X_train'), y_train=sup.get('y_train'),
    X_test=sup.get('X_test'), y_test=sup.get('y_test'),
)
if dl.get('mlp_trained'):
    mt = dl['mlp_test_metrics']
    print(f'MLP test: roc_auc={mt["roc_auc"]:.3f}, f1={mt["f1"]:.3f}')
else:
    print('MLP baseline skipped.')

In [ ]:
from src.evaluation import build_recommendations
combined = {
    'unsupervised': unsup,
    'supervised': sup,
    'deep_learning': dl,
    'eda': eda_summaries,
    'n_feature_customers': len(rfm_df),
    'rfm_df': rfm_df,
}
recs = build_recommendations(combined)
print(f'{len(recs)} recommendations built.')
for i, r in enumerate(recs[:5], start=1):
    print(f'[{i}] {r}')

## Production pipeline

For a complete end-to-end run, execute:
```bash
python -m src.main
```
which produces the full DOCX report, CSV tables, PNG figures and recommendations.